# 02 - Preprocessing

Applies the column decisions and cleaning made in `01_eda.ipynb`, then encodes categorical features and produces a train/test split.

Run this notebook with the **Python (Mental Health Prediction .venv)** kernel.

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = '../data/raw/Student Depression Dataset.csv'
df = pd.read_csv(DATA_PATH)
print('Raw shape:', df.shape)

Raw shape: (27901, 18)


## Apply cleaning and column decisions from EDA

- Drop 3 rows with missing `Financial Stress`
- Drop `id`, `City` (identifier / high-cardinality + data quality issues)
- Drop `Profession`, `Work Pressure`, `Job Satisfaction` (near-zero variance)
- `CGPA == 0` and `Age > 40` outliers: left as-is (too few rows to matter)

In [2]:
df = df.dropna(subset=['Financial Stress']).reset_index(drop=True)
df = df.drop(columns=['id', 'City', 'Profession', 'Work Pressure', 'Job Satisfaction'])
print('Shape after EDA decisions:', df.shape)
df.columns.tolist()

Shape after EDA decisions: (27898, 13)


['Gender',
 'Age',
 'Academic Pressure',
 'CGPA',
 'Study Satisfaction',
 'Sleep Duration',
 'Dietary Habits',
 'Degree',
 'Have you ever had suicidal thoughts ?',
 'Work/Study Hours',
 'Financial Stress',
 'Family History of Mental Illness',
 'Depression']

## Encode binary categorical columns

`Gender` (Male/Female) and `Family History of Mental Illness` (Yes/No) are mapped to 0/1.

In [3]:
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Family History of Mental Illness'] = (
    df['Family History of Mental Illness'].map({'Yes': 1, 'No': 0})
)

print(df['Gender'].value_counts())
print(df['Family History of Mental Illness'].value_counts())

Gender
1    15546
0    12352
Name: count, dtype: int64
Family History of Mental Illness
0    14397
1    13501
Name: count, dtype: int64


## Ordinal encode `Dietary Habits` and `Sleep Duration`

Both columns have a natural order. The small "Others" category (12 rows in Dietary Habits, 18 in Sleep Duration) has no signal to indicate which real category it belongs to, so it's mapped to the mode: `Unhealthy` for Dietary Habits, `Less than 5 hours` for Sleep Duration.

In [4]:
df['Dietary Habits'] = df['Dietary Habits'].replace('Others', 'Unhealthy')
dietary_map = {'Unhealthy': 0, 'Moderate': 1, 'Healthy': 2}
df['Dietary Habits'] = df['Dietary Habits'].map(dietary_map)

df['Sleep Duration'] = df['Sleep Duration'].replace('Others', 'Less than 5 hours')
sleep_map = {
    'Less than 5 hours': 0,
    '5-6 hours': 1,
    '7-8 hours': 2,
    'More than 8 hours': 3,
}
df['Sleep Duration'] = df['Sleep Duration'].map(sleep_map)

print(df['Dietary Habits'].value_counts().sort_index())
print(df['Sleep Duration'].value_counts().sort_index())

Dietary Habits
0    10328
1     9921
2     7649
Name: count, dtype: int64
Sleep Duration
0    8327
1    6181
2    7346
3    6044
Name: count, dtype: int64


## One-hot encode `Degree`

`Degree` has 28 unique values with no natural order, so it's one-hot encoded.

In [5]:
df = pd.get_dummies(df, columns=['Degree'], prefix='Degree')
print('Shape after one-hot encoding Degree:', df.shape)
df.columns.tolist()

Shape after one-hot encoding Degree: (27898, 40)


['Gender',
 'Age',
 'Academic Pressure',
 'CGPA',
 'Study Satisfaction',
 'Sleep Duration',
 'Dietary Habits',
 'Have you ever had suicidal thoughts ?',
 'Work/Study Hours',
 'Financial Stress',
 'Family History of Mental Illness',
 'Depression',
 'Degree_B.Arch',
 'Degree_B.Com',
 'Degree_B.Ed',
 'Degree_B.Pharm',
 'Degree_B.Tech',
 'Degree_BA',
 'Degree_BBA',
 'Degree_BCA',
 'Degree_BE',
 'Degree_BHM',
 'Degree_BSc',
 'Degree_Class 12',
 'Degree_LLB',
 'Degree_LLM',
 'Degree_M.Com',
 'Degree_M.Ed',
 'Degree_M.Pharm',
 'Degree_M.Tech',
 'Degree_MA',
 'Degree_MBA',
 'Degree_MBBS',
 'Degree_MCA',
 'Degree_MD',
 'Degree_ME',
 'Degree_MHM',
 'Degree_MSc',
 'Degree_Others',
 'Degree_PhD']

## Encode `Have you ever had suicidal thoughts ?`

Kept (strong relationship with the target, see EDA), encoded as binary 0/1.

In [6]:
df['Have you ever had suicidal thoughts ?'] = (
    df['Have you ever had suicidal thoughts ?'].map({'Yes': 1, 'No': 0})
)
df['Have you ever had suicidal thoughts ?'].value_counts()

Have you ever had suicidal thoughts ?
1    17656
0    10242
Name: count, dtype: int64

## Train/test split

80/20 split, stratified on `Depression` to preserve class balance, `random_state=42` for reproducibility.

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Depression'])
y = df['Depression']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)
print('Train class balance:')
print(y_train.value_counts(normalize=True))
print('Test class balance:')
print(y_test.value_counts(normalize=True))

Train shape: (22318, 39) | Test shape: (5580, 39)
Train class balance:
Depression
1    0.585536
0    0.414464
Name: proportion, dtype: float64
Test class balance:
Depression
1    0.585484
0    0.414516
Name: proportion, dtype: float64


## Save processed data

In [8]:
train = X_train.copy()
train['Depression'] = y_train
test = X_test.copy()
test['Depression'] = y_test

train.to_csv('../data/processed/train.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)
print('Saved train.csv and test.csv to data/processed/')

Saved train.csv and test.csv to data/processed/
